<a href="https://colab.research.google.com/github/AarnavSawant/KVCompass/blob/main/KVCompass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KVCompass Demo Notebook

This notebook is tuned for a professor demo: define a sweep in YAML, run it once, then surface the most important quality and runtime results in clean tables and plots.


In [ ]:
# Clone or refresh the repo, switch to the sweep-workflow branch, then install the project.
from pathlib import Path

repo_dir = Path('/content/KVCompass')
if not repo_dir.exists():
    !git clone https://github.com/AarnavSawant/KVCompass.git /content/KVCompass

%cd /content/KVCompass
!git fetch origin
!git checkout codex/colab-sweep-workflow
!git pull
!nvidia-smi
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .


In [ ]:
# Set HF_TOKEN from Colab secrets, then verify the login.
from google.colab import userdata
from huggingface_hub import whoami
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print(whoami())


In [ ]:
# Edit this YAML to choose the benchmark scenarios, methods, and budgets you want.
# The default set avoids quant_4bit because Colab T4 GPUs usually fail on the quanto CUDA backend.
from pathlib import Path

sweep_yaml = '''
sweep:
  name: colab_demo_ruler
  model: Qwen/Qwen2.5-1.5B-Instruct
  device: auto
  torch_dtype: bfloat16
  methods_config_path: configs/methods.yaml
  output_dir: results/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: retrieval_long
      dataset: ruler
      data_dir: "4096"
      fraction: 0.005
      methods:
        - no_compression
        - snapkv
        - expected_attention
        - adakv_expected
        - pyramidkv
        - streaming_llm
      budgets:
        no_compression: [1.0]
        default: [0.5]
    - name: memory_tight
      dataset: ruler
      data_dir: "4096"
      fraction: 0.005
      methods:
        - snapkv
        - expected_attention
        - adakv_expected
        - knorm
        - tova
        - random
      budgets:
        default: [0.25]
'''

config_path = Path('configs/benchmark_sweeps.colab.yaml')
config_path.write_text(sweep_yaml, encoding='utf-8')
print(config_path.read_text())


In [ ]:
# Run every method listed in the YAML with one command.
# The generic transformers message about using a dataset can still appear,
# but this benchmark path does load a Hugging Face dataset. The run groups
# requests by shared context so KVPress can reuse the compressed KV cache.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.colab.yaml


In [ ]:
# Inspect the latest sweep summary, plus a compact demo-ready leaderboard.
import json
from pathlib import Path
import pandas as pd

summary_files = sorted(Path("results/benchmark_eval").glob("*__summary.csv"))
latest_summary = summary_files[-1]
summary_df = pd.read_csv(latest_summary)
print("SUMMARY:", latest_summary)
display(summary_df)

rows = []
for _, row in summary_df.iterrows():
    metrics = json.loads(Path(row["metrics_path"]).read_text())
    metric_values = []
    for task_metrics in metrics.values():
        if isinstance(task_metrics, dict):
            metric_values.extend(v for v in task_metrics.values() if isinstance(v, (int, float)))
    rows.append({
        "method": row["method"],
        "budget": row["budget"],
        "avg_quality": round(sum(metric_values) / len(metric_values), 2) if metric_values else None,
        "avg_latency_seconds": row.get("avg_latency_seconds"),
        "avg_throughput_tokens_per_second": row.get("avg_throughput_tokens_per_second"),
        "peak_gpu_memory_mb": row.get("peak_gpu_memory_mb"),
    })

leaderboard_df = pd.DataFrame(rows).sort_values(["avg_quality", "avg_latency_seconds"], ascending=[False, True])
print("\nDemo Leaderboard")
display(leaderboard_df)

latest_run = max(Path("results/benchmark_eval").glob("*/*"), key=lambda p: p.stat().st_mtime).parent
print("\nLATEST RUN:", latest_run)
print("\nmetrics.json")
print((latest_run / "metrics.json").read_text())
print("\nrun_stats.json")
print((latest_run / "run_stats.json").read_text())


In [ ]:
# Plot the main tradeoffs for the latest sweep.
import matplotlib.pyplot as plt
import pandas as pd

plot_df = leaderboard_df.copy()
plot_df["label"] = plot_df["method"] + " @ " + plot_df["budget"].astype(str)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].bar(plot_df["label"], plot_df["avg_quality"], color="#2a6f97")
axes[0].set_title("Average Benchmark Quality")
axes[0].tick_params(axis="x", rotation=60)

axes[1].bar(plot_df["label"], plot_df["avg_latency_seconds"], color="#c1121f")
axes[1].set_title("Average Latency (s)")
axes[1].tick_params(axis="x", rotation=60)

axes[2].bar(plot_df["label"], plot_df["peak_gpu_memory_mb"], color="#6a994e")
axes[2].set_title("Peak GPU Memory (MB)")
axes[2].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()


In [ ]:
# Build a simple recommendation table for the demo.
best_quality = leaderboard_df.sort_values(["avg_quality", "avg_latency_seconds"], ascending=[False, True]).iloc[0]
best_latency = leaderboard_df.sort_values("avg_latency_seconds", ascending=True).iloc[0]
best_memory = leaderboard_df.dropna(subset=["peak_gpu_memory_mb"]).sort_values("peak_gpu_memory_mb", ascending=True).iloc[0] if leaderboard_df["peak_gpu_memory_mb"].notna().any() else None
balanced_df = leaderboard_df.copy()
for col in ["avg_quality", "avg_latency_seconds"]:
    vals = balanced_df[col].astype(float)
    if col == "avg_quality":
        balanced_df[col + "_norm"] = (vals - vals.min()) / (vals.max() - vals.min() + 1e-8)
    else:
        balanced_df[col + "_norm"] = 1 - ((vals - vals.min()) / (vals.max() - vals.min() + 1e-8))
balanced_df["balanced_score"] = 0.6 * balanced_df["avg_quality_norm"] + 0.4 * balanced_df["avg_latency_seconds_norm"]
best_balanced = balanced_df.sort_values("balanced_score", ascending=False).iloc[0]

rec_rows = [
    {"category": "best_for_quality", "method": best_quality["method"], "budget": best_quality["budget"]},
    {"category": "best_for_latency", "method": best_latency["method"], "budget": best_latency["budget"]},
]
if best_memory is not None:
    rec_rows.append({"category": "best_for_memory", "method": best_memory["method"], "budget": best_memory["budget"]})
rec_rows.append({"category": "best_balanced", "method": best_balanced["method"], "budget": best_balanced["budget"]})
recommendation_df = pd.DataFrame(rec_rows)
display(recommendation_df)


In [ ]:
# Optional: run one benchmark directly instead of the whole sweep.
!python scripts/run_kvpress_benchmark_eval.py \
  --dataset ruler \
  --data-dir 4096 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --method no_compression \
  --budget 1.0 \
  --fraction 0.002 \
  --torch-dtype bfloat16
